# Problema #02 — Cortes más chicos y mezcla de métodos (Grupo A)

**Objetivo: llegar al mejor puntaje posible en Kaggle.** Este notebook **no entrena nada**: usa las probabilidades que ya guardaron las corridas `WF96xx` (10 semillas por método) para probar dos ideas baratas:

1. **Cortes más chicos.** En todos los métodos el puntaje de Kaggle baja al subir el corte, y el mejor valor está en el corte más chico que probamos (1600–1800). El óptimo puede estar por debajo de 1600.
2. **Mezclar métodos.** Promediar los resultados de varios tratamientos del drifting (por ejemplo `ninguno` + `estandarizar`) puede dar un ranking más estable que cualquiera por separado.

**Qué hace**
* Arma cada *candidato* (un método solo, o una mezcla de varios) a partir de las probabilidades guardadas.
* Calcula su **ganancia en validación local (202107)** para muchos cortes. Es solo una guía: ya vimos que la validación local y Kaggle no siempre coinciden.
* Genera los archivos de Kaggle (`KB_<candidato>_<corte>.csv`) y, si se lo pedís, **sube solo los que elijas**, para controlar el cupo de 100 envíos por día.

**Salidas** (en `PARAM$carpeta_salida`): `resumen_local.csv`, `curva_local.csv`, `solapamiento_metodos.csv`, `curva_local.png`, y la carpeta `kaggle/`.

## 1. Inicialización y parámetros

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

require("data.table")

In [ ]:
PARAM <- list()

# donde estan las carpetas WF96xx
PARAM$carpeta_exp <- "/content/buckets/b1/exp"
PARAM$experimento_base <- 9600

# MISMO orden que metodos_DR del notebook 06 (define el numero de carpeta)
metodos_DR <- c(
  "ninguno", "deflacion", "dolar_blue", "dolar_oficial",
  "rank_simple", "rank_cero_fijo", "estandarizar",
  "ninguno_pond", "deflacion_pond",
  "deflacion_selectiva"
)

PARAM$semillas <- c(
  165463, 329969, 454253, 593899, 980249,
  100183, 250027, 400031, 550049, 700057
)

PARAM$carpeta_salida <- file.path(PARAM$carpeta_exp, paste0("BLEND_", PARAM$experimento_base))

# como se combinan los metodos dentro de una mezcla:
#  "prob" = promedio de las probabilidades | "rank" = promedio de los rankings (percentiles)
PARAM$combinar <- "prob"

# candidatos: nombre = c(metodos que lo forman). Un solo metodo = ese metodo solo.
#  (si falta alguno, el candidato se saltea y se avisa)
PARAM$candidatos <- list(
  ninguno        = "ninguno",
  estandarizar   = "estandarizar",
  dolar_oficial  = "dolar_oficial",
  rank_cero_fijo = "rank_cero_fijo",
  mezcla_ninguno_estandarizar = c("ninguno", "estandarizar"),
  mezcla_4 = c("ninguno", "estandarizar", "dolar_oficial", "rank_cero_fijo"),
  mezcla_6 = c("ninguno", "dolar_blue", "dolar_oficial", "rank_cero_fijo", "rank_simple", "estandarizar")
)

# cortes (cantidad de envios sobre los clientes de 202109) de los archivos de Kaggle a generar
PARAM$cortes_kaggle <- seq(1000, 2200, by = 100)

# cortes de la curva local (se llevan a la proporcion de 202107)
PARAM$cortes_local <- seq(600, 3000, by = 100)

# rango de cortes del enunciado (Grupo A) para el promedio de referencia
PARAM$rango_enunciado <- c(1800, 2400)

# ganancia: solo BAJA+2 suma (el costo -25000 es un SUPUESTO)
PARAM$ganancia$acierto <- 975000
PARAM$ganancia$costo <- -25000

PARAM$kaggle$competencia <- "labo-1-virtual-2026-analista-jr"

dir.create(PARAM$carpeta_salida, showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(PARAM$carpeta_salida, "kaggle"), showWarnings = FALSE)

stopifnot(PARAM$combinar %in% c("prob", "rank"))

## 2. Lectura de las probabilidades de cada método
Por método: la probabilidad de validación (promedio de las 10 semillas, 202107) y la del futuro (`prediccion.txt`, 202109).

In [ ]:
carpeta_metodo <- function(m) {
  file.path(PARAM$carpeta_exp, paste0("WF", PARAM$experimento_base + match(m, metodos_DR) - 1))
}

leer_metodo <- function(m) {
  carpeta <- carpeta_metodo(m)
  f_val <- file.path(carpeta, paste0("prediccion_val_", PARAM$semillas, ".txt"))
  f_fut <- file.path(carpeta, "prediccion.txt")
  f_cla <- file.path(carpeta, "validacion_clases.txt")
  if (!all(file.exists(c(f_val, f_fut, f_cla)))) return(NULL)

  val <- lapply(f_val, fread)
  ids <- val[[1]]$numero_de_cliente
  stopifnot(all(vapply(val, function(d) identical(d$numero_de_cliente, ids), logical(1))))

  list(
    val = data.table(numero_de_cliente = ids, prob = rowMeans(vapply(val, function(d) d$prob, numeric(length(ids))))),
    fut = fread(f_fut),
    clases = fread(f_cla)
  )
}

nombres_metodos <- unique(unlist(PARAM$candidatos))
stopifnot(all(nombres_metodos %in% metodos_DR))

datos <- lapply(setNames(nombres_metodos, nombres_metodos), leer_metodo)
faltan <- names(datos)[vapply(datos, is.null, logical(1))]
if (length(faltan) > 0) cat("metodos sin corrida completa (se ignoran los candidatos que los usan):", faltan, "\n")
datos <- datos[!vapply(datos, is.null, logical(1))]

# todas las corridas tienen que tener los mismos clientes (misma foto_mes) y las mismas clases
ref <- datos[[1]]
for (m in names(datos)) {
  stopifnot(
    setequal(datos[[m]]$val$numero_de_cliente, ref$val$numero_de_cliente),
    setequal(datos[[m]]$fut$numero_de_cliente, ref$fut$numero_de_cliente)
  )
}
clases <- ref$clases[match(ref$val$numero_de_cliente, numero_de_cliente), clase_ternaria]
cat("clientes validacion:", nrow(ref$val), "| BAJA+2:", sum(clases == "BAJA+2"),
  "| clientes futuro:", nrow(ref$fut), "\n")

## 3. Armado de los candidatos (un método o una mezcla)

In [ ]:
puntaje <- function(tb, ids) {
  p <- tb$prob[match(ids, tb$numero_de_cliente)]
  if (PARAM$combinar == "rank") p <- frank(p, ties.method = "average") / length(p)
  p
}

armar_candidato <- function(metodos) {
  list(
    val = rowMeans(vapply(metodos, function(m) puntaje(datos[[m]]$val, ref$val$numero_de_cliente),
      numeric(nrow(ref$val)))),
    fut = rowMeans(vapply(metodos, function(m) puntaje(datos[[m]]$fut, ref$fut$numero_de_cliente),
      numeric(nrow(ref$fut))))
  )
}

candidatos <- PARAM$candidatos[vapply(PARAM$candidatos, function(ms) all(ms %in% names(datos)), logical(1))]
stopifnot(length(candidatos) > 0)
cand <- lapply(candidatos, armar_candidato)
names(cand)

## 4. Ganancia local (validación 202107) para muchos cortes
Los cortes se escalan a la población de 202107. **Es una guía, no la verdad**: en el baseline la validación local y Kaggle ordenaron los cortes al revés.

In [ ]:
ganancia_cortes <- function(score, clases, cortes_val) {
  orden <- order(score, decreasing = TRUE)
  gan <- ifelse(clases[orden] == "BAJA+2", PARAM$ganancia$acierto, PARAM$ganancia$costo)
  cumsum(gan)[pmin(cortes_val, length(gan))]
}

escala <- nrow(ref$val) / nrow(ref$fut)
cortes_val <- pmax(1L, as.integer(round(PARAM$cortes_local * escala)))

curva <- rbindlist(lapply(names(cand), function(n) {
  data.table(candidato = n, corte = PARAM$cortes_local,
    ganancia_M = ganancia_cortes(cand[[n]]$val, clases, cortes_val) / 1e6)
}))
fwrite(curva, file.path(PARAM$carpeta_salida, "curva_local.csv"))

resumen <- curva[, list(
  mejor_corte_local = corte[which.max(ganancia_M)],
  ganancia_max_M = max(ganancia_M),
  prom_cortes_enunciado_M = mean(ganancia_M[corte >= PARAM$rango_enunciado[1] & corte <= PARAM$rango_enunciado[2]]),
  prom_cortes_1000_2200_M = mean(ganancia_M[corte >= 1000 & corte <= 2200])
), by = candidato]
setorder(resumen, -prom_cortes_enunciado_M)
fwrite(resumen, file.path(PARAM$carpeta_salida, "resumen_local.csv"))
resumen[, (names(resumen)[-1]) := lapply(.SD, round, 2), .SDcols = names(resumen)[-1]]
resumen

In [ ]:
# curva local por candidato
graficar <- function() {
  w <- dcast(curva, corte ~ candidato, value.var = "ganancia_M")
  cols <- seq_along(cand)
  matplot(w$corte, as.matrix(w[, -1]), type = "l", lty = 1, lwd = 2, col = cols,
    xlab = "cortes (envios sobre 202109)", ylab = "ganancia local 202107 (M)",
    main = "Ganancia local por corte")
  abline(v = PARAM$rango_enunciado, lty = 2)
  legend("bottomleft", legend = names(cand), col = cols, lty = 1, lwd = 2, cex = 0.7)
}
graficar()
png(file.path(PARAM$carpeta_salida, "curva_local.png"), width = 1000, height = 600)
graficar()
invisible(dev.off())

## 5. ¿Cuánto se parecen los métodos? (solapamiento de los primeros envíos)
Fracción de clientes que dos métodos comparten entre sus primeros N en 202109. Cuanto menos se parecen, más aporta mezclarlos.

In [ ]:
N_solap <- 1800
tops <- lapply(cand[names(cand) %in% nombres_metodos], function(x) ref$fut$numero_de_cliente[order(-x$fut)[1:N_solap]])
nm <- names(tops)
solap <- data.table(expand.grid(a = nm, b = nm, stringsAsFactors = FALSE))
solap[, solapamiento := mapply(function(a, b) length(intersect(tops[[a]], tops[[b]])) / N_solap, a, b)]
fwrite(solap, file.path(PARAM$carpeta_salida, "solapamiento_metodos.csv"))
m_solap <- dcast(solap, a ~ b, value.var = "solapamiento")
m_solap[, (names(m_solap)[-1]) := lapply(.SD, round, 3), .SDcols = names(m_solap)[-1]]
m_solap

## 6. Archivos de Kaggle (no se sube nada todavía)
`KB_<candidato>_<corte>.csv`: marca como `Predicted = 1` los primeros `corte` clientes del candidato en 202109.

In [ ]:
for (n in names(cand)) {
  orden <- order(cand[[n]]$fut, decreasing = TRUE)
  ids_ordenados <- ref$fut$numero_de_cliente[orden]
  for (corte in PARAM$cortes_kaggle) {
    tb <- data.table(numero_de_cliente = ref$fut$numero_de_cliente, Predicted = 0L)
    tb[numero_de_cliente %in% ids_ordenados[seq_len(corte)], Predicted := 1L]
    fwrite(tb, file.path(PARAM$carpeta_salida, "kaggle", paste0("KB_", n, "_", corte, ".csv")))
  }
}
cat("archivos generados:", length(list.files(file.path(PARAM$carpeta_salida, "kaggle"))), "\n")

## 7. Subir a Kaggle SOLO lo que elijas
Completá `PARAM$enviar` con los pares (candidato, corte) que quieras subir. **Cada fila es un envío del cupo diario (100).** Con la tabla vacía no se sube nada. Mirá antes `resumen_local.csv` y el gráfico para elegir.

Ejemplo: `data.table(candidato = "estandarizar", corte = c(1200, 1400, 1600))`

In [ ]:
PARAM$enviar <- data.table(candidato = character(), corte = integer())

In [ ]:
stopifnot(nrow(PARAM$enviar) <= 100)

for (i in seq_len(nrow(PARAM$enviar))) {
  n <- PARAM$enviar$candidato[i]
  corte <- PARAM$enviar$corte[i]
  archivo <- file.path(PARAM$carpeta_salida, "kaggle", paste0("KB_", n, "_", corte, ".csv"))
  if (!file.exists(archivo)) { cat("no existe", archivo, "\n"); next }

  linea <- paste("kaggle competitions submit -c", PARAM$kaggle$competencia, "-f", shQuote(archivo),
    "-m", shQuote(paste0("BLEND=", n, " (", PARAM$combinar, ")  ensamble=10sem  envios=", corte)))

  salida <- tryCatch(system(linea, intern = TRUE),
    error = function(e) conditionMessage(e), warning = function(w) conditionMessage(w))
  Sys.sleep(30)
  cat(n, corte, ":", salida, "\n")
}
cat("envios realizados:", nrow(PARAM$enviar), "\n")